# 8. Benchmark chính thức Milestone 2 trên Google Colab CPU-only

## 8.1 Môi trường đo và tổng hợp 5 phiên liên tiếp

Toàn bộ **5 phiên full** (mỗi phiên chạy cold **5.000 bài**, `reused=0`) được thực thi tuần tự trên **Google Colab Standard CPU-only** (`is_google_colab=True`, `cpu_only=True`, 2 logical vCPU Intel x86_64, RAM 12.975,52 MB, Python 3.13.18) và tổng hợp bằng `src/crawler/benchmark_sessions.py`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2, bench = data['scope'], data['m1'], data['m2'], data['m2_benchmark']
plt.rcParams.update({'figure.figsize': (9.2, 4.2), 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 120})
print(f"Toàn nhóm (5 thành viên): {scope['papers']:,} paper / {scope['versions']:,} version | Cập nhật: {data['report_date']}")

In [ ]:
display(pd.Series(bench['environment'], name='Giá trị').to_frame())
tot = bench['totals']
display(Markdown('**Bảng 8.1. Tổng hợp benchmark Milestone 2 trên Google Colab CPU-only (25.000 bài).**'))
display(pd.DataFrame([
    ('Mọi phiên đều là Colab CPU-only', str(tot['all_sessions_colab_cpu'])),
    ('Phiên chạy gối giờ / trùng file stats', f"{tot['sessions_overlap']} / {tot['duplicate_run_files_skipped']}"),
    ('Tổng bài parse mới (cold, reused=0)', f"{tot['unique_publications']:,} bài"),
    ('Tổng thời gian xử lý + validate (5 phiên)', f"{tot['total_profiler_sec']:,.3f} giây (~{tot['total_profiler_sec']/3600:.2f} giờ)"),
    ('Thời gian trung bình mỗi publication', f"{tot['avg_sec_per_unique_publication']:.4f} giây/bài"),
    ('Peak RSS lớn nhất quan sát được', f"{tot['max_session_peak_rss_mib']:.2f} MiB"),
    ('RSS trung bình trọng số thời gian', f"{tot['session_time_weighted_avg_rss_mib']:.2f} MiB"),
    ('Peak disk workspace lớn nhất', f"{tot['max_session_peak_disk_bytes']/1024**3:.2f} GiB"),
    ('Dung lượng 3 bảng Parquet chuẩn hợp nhất', f"{tot['final_output_bytes']:,} byte ({tot['final_output_bytes']/1024**3:.2f} GiB)"),
], columns=['Chỉ tiêu', 'Kết quả đo']))
df_sess = pd.DataFrame(bench['sessions'])
display(Markdown('**Bảng 8.2. Chi tiết từng phiên Colab CPU của 5 thành viên.**'))
display(df_sess[['session', 'processed_attempts', 'profiler_sec', 'avg_sec_per_pub', 'peak_rss_mib', 'avg_rss_mib', 'final_output_bytes']])

In [ ]:
phases = pd.DataFrame(bench['phases_aggregated']).T.sort_values('elapsed_sec', ascending=False)
display(Markdown('**Bảng 8.3. Phân bổ thời gian các pha xử lý M2 (cộng dồn 5 phiên).**'))
display(phases)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df_s = pd.DataFrame(bench['sessions'])
axes[0].bar(df_s['member'], df_s['avg_sec_per_pub'], color='#2f628e')
axes[0].axhline(bench['totals']['avg_sec_per_unique_publication'], color='#b5534a', linestyle='--', label='TB chung 0,748s/bài')
axes[0].set(ylabel='Giây / publication', title='Tốc độ trung bình từng thành viên (Colab CPU)')
axes[0].legend(fontsize=8)
axes[1].bar(df_s['member'], df_s['peak_rss_mib'], color='#b57b36', label='Peak RSS (MiB)')
axes[1].bar(df_s['member'], df_s['avg_rss_mib'], color='#34785a', label='Mean RSS (MiB)')
axes[1].set(ylabel='MiB', title='Mức sử dụng RAM trên Colab CPU')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
profiles = bench.get('resource_profiles') or {}
avail = {k: v for k, v in profiles.items() if v and v.get('rows')}
if avail:
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.2))
    for label, prof in avail.items():
        f = pd.DataFrame(prof['rows'])
        t_min = f['window_end_sec'] / 60.0
        axes[0].plot(t_min, f['rss_sample_mean_mib'], label=label, linewidth=1.3)
        disk_rows = f.dropna(subset=['disk_peak_bytes'])
        if not disk_rows.empty:
            axes[1].plot(disk_rows['window_end_sec'] / 60.0, disk_rows['disk_peak_bytes'] / 1024**3, label=label, linewidth=1.3)
    axes[0].set(xlabel='Phút chạy', ylabel='Mean RSS trong cửa sổ 60s (MiB)', title='Diễn biến RAM theo thời gian (Colab CPU)')
    axes[0].legend(fontsize=8)
    axes[1].set(xlabel='Phút chạy', ylabel='Disk workspace (GiB)', title='Dung lượng workspace parsed theo thời gian')
    axes[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()
    display(Markdown('**Hình 8.2. Chuỗi thời gian RSS và đĩa đo từ `resource_samples.jsonl` (4 phiên bật sync đầy đủ: TUAN, TAN, TRI, TRIET; cửa sổ 60s).**'))
display(Markdown(f"Dung lượng đĩa đỉnh của workspace trên Google Colab CPU-only đạt **{bench['totals']['max_session_peak_disk_bytes']/1024**3:.2f} GiB** (`{bench['totals']['max_session_peak_disk_bytes']:,}` byte) và 3 bảng Parquet đầu ra cuối còn **{bench['totals']['final_output_bytes']/1024**3:.2f} GiB** (`{bench['totals']['final_output_bytes']/1e9:.2f} GB`), nằm trong giới hạn đĩa của Colab."))

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức M2](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html) & [Terms of Use](https://info.arxiv.org/help/api/tou.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [src/parser/main.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/main.py).
- Mã nguồn triển khai: [src/crawler/profiler.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/profiler.py).
- Mã nguồn triển khai: [src/crawler/benchmark_sessions.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/benchmark_sessions.py).
- Mã nguồn triển khai: [tools/m2_colab_runner.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_colab_runner.py).